# Metacatalog VLSSR cross-match QA

Post-hoc QA against the VLSSR reference catalog at ~73 MHz (Blue band).

**Primary metric:** Blue-associated metacatalog completeness — fraction of rows with `"Blue" in bands_present` that match ≥1 VLSSR source within beam.

**Failure-mode metric:** VLSSR over-splitting — one VLSSR source matched by multiple metacatalog rows (possible LST/band over-split).

Matching uses primary metacatalog `RA`/`DEC` and `BMAJ_match` via `resolve_bmaj`, not Blue-band column positions.

**Run cells in order.**

In [1]:
from pathlib import Path

import pandas as pd

from lwa_catalog import CatalogLayout, read_metacatalog
from lwa_catalog.analyze import (
    VlssrMatchConfig,
    match_catalog_to_vlssr,
    select_blue_associated_rows,
    summarize_vlssr_match,
)
from lwa_catalog.constants import VLSSR_DEFAULT_PATH

# --- operator config ---
CATALOG_DIR = Path("/fast/claw/metacatalog_coadd2")  # existing fusion tree
VLSSR_PATH = VLSSR_DEFAULT_PATH  # override if VLSSR lives elsewhere
RUN_LST_MERGED_PASS = False  # optional ad-hoc pre-fusion check (not required for v1)

layout = CatalogLayout(CATALOG_DIR)
config = VlssrMatchConfig(catalog_path=VLSSR_PATH)

print("CATALOG_DIR =", layout.root.resolve())
print("VLSSR_PATH =", Path(VLSSR_PATH).resolve())
print("RUN_LST_MERGED_PASS =", RUN_LST_MERGED_PASS)

CATALOG_DIR = /fast/claw/metacatalog_coadd2
VLSSR_PATH = /fast/claw/vlssr_radecpeak.txt
RUN_LST_MERGED_PASS = False


## Load metacatalog

In [2]:
metacatalog = read_metacatalog(layout)
blue_meta = select_blue_associated_rows(metacatalog)

print(f"metacatalog rows: {len(metacatalog)}")
print(f"Blue-associated rows: {len(blue_meta)}")

metacatalog rows: 51407
Blue-associated rows: 49956


## Cross-match against VLSSR

In [3]:
result = match_catalog_to_vlssr(metacatalog, config=config)
print(summarize_vlssr_match(result))

LWA target rows:               49956
VLSSR footprint (Dec box):     92965
Meta matched (>=1 VLSSR):      42867
Blue completeness:            0.858
VLSSR matched (>=1 meta):      62132
VLSSR recovery:               0.668
VLSSR over-split (n_meta>1):   11760
Meta multi-VLSSR (n_vlssr>1):  19078
Max VLSSR hits per meta:          20


## Diagnostics

In [4]:
meta_flags = result.meta_flags
vlssr_flags = result.vlssr_flags

print("VLSSR hits per meta (value_counts):")
display(meta_flags["n_vlssr"].value_counts().sort_index())

print("\nIncomplete Blue-associated meta rows (no VLSSR match, first 20):")
display(meta_flags.loc[~meta_flags["matched"]].head(20))

print("\nOver-split VLSSR rows (multiple meta matches, first 20):")
display(vlssr_flags.loc[vlssr_flags["oversplit"]].head(20))

VLSSR hits per meta (value_counts):


n_vlssr
0      7089
1     23789
2     10736
3      5020
4      2139
5       772
6       267
7        93
8        24
9        12
10        5
11        4
13        2
14        1
17        2
20        1
Name: count, dtype: int64


Incomplete Blue-associated meta rows (no VLSSR match, first 20):


,meta_id,RA,DEC,n_vlssr,matched
57,27780,302.702635,43.262970,0,False
79,27781,270.079278,-23.268932,0,False
83,85,269.899660,-23.591657,0,False
86,88,201.405415,-42.844084,0,False
90,27782,305.775648,40.399220,0,False
140,139,298.781103,42.337192,0,False
172,170,345.501514,58.844387,0,False
201,202,352.936343,56.709416,0,False
216,216,60.748191,45.099294,0,False
230,230,303.174304,39.786369,0,False



Over-split VLSSR rows (multiple meta matches, first 20):


,vlssr_pos,RA,DEC,Peak_flux,n_meta,oversplit
3,3,0.007936,38.994399,0.720755,2,True
26,26,0.082138,81.401267,0.922460,2,True
28,28,0.085720,55.652155,15.599744,2,True
29,29,0.088241,78.671224,1.347578,2,True
35,35,0.101020,12.495618,1.194725,2,True
36,36,0.103249,44.152582,0.686972,4,True
54,54,0.135903,60.343655,2.261703,2,True
56,56,0.153266,12.204518,0.919142,2,True
61,61,0.166437,43.951383,1.151125,2,True
64,64,0.171105,39.297216,0.772535,2,True


## Optional: LST-merged Blue pass

Ad-hoc pre-fusion check. Not required for v1 sign-off. Set `RUN_LST_MERGED_PASS = True` in the config cell.

In [ ]:
if RUN_LST_MERGED_PASS:
    from lwa_catalog.io import read_lst_merged

    lst_blue = read_lst_merged(layout, "Blue")
    lst_config = VlssrMatchConfig(catalog_path=VLSSR_PATH, target="lst_merged_blue")
    lst_result = match_catalog_to_vlssr(lst_blue, config=lst_config)
    print("LST-merged Blue pass:")
    print(summarize_vlssr_match(lst_result))
else:
    print("Skipping LST-merged Blue pass (RUN_LST_MERGED_PASS=False)")